In [76]:
import pandas as pd
import numpy as np
import joblib

In [77]:
df = pd.read_csv("house_data.csv")

In [78]:
df.head()

,size_sqft,bedrooms,price
0,800.0,1.0,3000000
1,900.0,2.0,3500000
2,1000.0,2.0,4000000
3,1100.0,3.0,4500000
4,1200.0,3.0,5000000


In [79]:
df.columns = df.columns.str.strip()
df = df.drop_duplicates()

In [80]:
from sklearn.model_selection import train_test_split
X = df[["size_sqft","bedrooms"]]
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [81]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [82]:
size = ["size_sqft"]
room = ["bedrooms"]

In [83]:
pipe_size = Pipeline([
    ("imputer",SimpleImputer(strategy="mean")),
    ("scaler",StandardScaler())
])
pipe_room = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent"))
])

In [84]:
preprocessor = ColumnTransformer([
    ("size",pipe_size,size),
    ("room",pipe_room,room)
])

In [85]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

In [86]:
model_linreg = Pipeline([
    ("preprocessor",preprocessor),
    ("model",LinearRegression())
])
model_tree = Pipeline([
    ("preprocessor",preprocessor),
    ("model",DecisionTreeRegressor(random_state=42))
])
model_svr = Pipeline([
    ("preprocessor",preprocessor),
    ("model",SVR())
])

In [87]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [88]:
cv = StratifiedKFold(n_splits=2,shuffle=True,random_state=42)

scoring = {
    'MAE':'neg_mean_absolute_error',
    'MSE': 'neg_mean_squared_error',
    'R2': 'r2'
}

scores_lin_reg = cross_validate(
    model_linreg,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)
maelr = scores_lin_reg['test_MAE'].mean()
mselr = scores_lin_reg['test_MSE'].mean()
r2lr = scores_lin_reg['test_R2'].mean()

scores_tree = cross_validate(
    model_tree,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)
maet = scores_tree['test_MAE'].mean()
mset = scores_tree['test_MSE'].mean()
r2t = scores_tree['test_R2'].mean()

scores_svr = cross_validate(
    model_svr,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)
maes = scores_svr['test_MAE'].mean()
mses = scores_svr['test_MSE'].mean()
r2s = scores_svr['test_R2'].mean()


C:\Users\SAURODEEP DE\AppData\Roaming\Python\Python311\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(
C:\Users\SAURODEEP DE\AppData\Roaming\Python\Python311\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(
C:\Users\SAURODEEP DE\AppData\Roaming\Python\Python311\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(


In [89]:
r2_s = [r2lr, r2t, r2s]

for i in r2_s:
    if i == max(r2_s):
        if i == r2lr:
            best_model = model_linreg
            print("Best model is Linear Regression with R2 score of {:.2f}".format(i))
        elif i == r2t:
            best_model = model_tree
            print("Best model is Decision Tree with R2 score of {:.2f}".format(i))
        else:
            best_model = model_svr
            print("Best model is Support Vector Regression with R2 score of {:.2f}".format(i))

Best model is Decision Tree with R2 score of 0.82


In [90]:
print("Linear Regression: MAE = {:.2f}, MSE = {:.2f}".format(-maelr, -mselr))
print("Decision Tree: MAE = {:.2f}, MSE = {:.2f}".format(-maet, -mset))
print("Support Vector Regression: MAE = {:.2f}, MSE = {:.2f}".format(-maes, -mses))

Linear Regression: MAE = 292242.38, MSE = 257663423502.79
Decision Tree: MAE = 409113.16, MSE = 280859732631.58
Support Vector Regression: MAE = 1128752.74, MSE = 1815833406787.91


In [91]:
model_tree.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('size', ...), ('room', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers 

In [92]:
joblib.dump(model_tree, "best_model.pkl")

['best_model.pkl']